In [1]:
import sqlite3
import hashlib
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


def extract_guids(url_string):
    if not url_string:
        return []
    url_string = url_string.strip("[]").replace("'", "").replace('"', "")
    if not url_string.strip():
        return []
    urls = [url.strip() for url in url_string.split(',')]
    return [url.split('/')[-1] for url in urls if url.strip()]


def execute_link_load(conn, cursor, table_name, rows, insert_sql):
    try:
        cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
        before_count = cursor.fetchone()[0]

        cursor.executemany(insert_sql, rows)
        conn.commit()

        cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
        after_count = cursor.fetchone()[0]

        inserted = after_count - before_count
        skipped = len(rows) - inserted

        if len(rows) == 0:
            message = f"{table_name}: {before_count} rows already in DB | no source rows to process"
        else:
            message = f"{table_name}: {before_count} rows already in DB | {inserted} rows inserted | {skipped} rows skipped (already existed) | {after_count} total rows now in DB"

        print(message)
        logger.info(message)
        return inserted, True

    except Exception as e:
        conn.rollback()
        error_message = f"{table_name} FAILED and rolled back: {str(e)}"
        print(error_message)
        logger.error(error_message)
        return 0, False


def run_link_loads(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    startup_message = f"Connected to: {db_path}"
    print(startup_message)
    logger.info(startup_message)

    total_inserted = 0
    failed_tables = []

    cursor.execute('SELECT id, appearances FROM stg_bosses WHERE appearances IS NOT NULL')
    rows = [(boss_id, game_id) for boss_id, appearances in cursor.fetchall() for game_id in extract_guids(appearances)]
    inserted, success = execute_link_load(conn, cursor, "link_boss_games", rows, 'INSERT OR IGNORE INTO "link_boss_games" ("Boss_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_boss_games")

    cursor.execute('SELECT id, dungeons FROM stg_bosses WHERE dungeons IS NOT NULL')
    rows = [(boss_id, dungeon_id) for boss_id, dungeons in cursor.fetchall() for dungeon_id in extract_guids(dungeons)]
    inserted, success = execute_link_load(conn, cursor, "link_boss_dungeons", rows, 'INSERT OR IGNORE INTO "link_boss_dungeons" ("Boss_ID", "Dungeon_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_boss_dungeons")

    cursor.execute('SELECT id, appearances FROM stg_characters WHERE appearances IS NOT NULL')
    rows = [(char_id, game_id) for char_id, appearances in cursor.fetchall() for game_id in extract_guids(appearances)]
    inserted, success = execute_link_load(conn, cursor, "link_character_games", rows, 'INSERT OR IGNORE INTO "link_character_games" ("Character_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_character_games")

    cursor.execute('SELECT id, appearances FROM stg_dungeons WHERE appearances IS NOT NULL')
    rows = [(dungeon_id, game_id) for dungeon_id, appearances in cursor.fetchall() for game_id in extract_guids(appearances)]
    inserted, success = execute_link_load(conn, cursor, "link_dungeon_games", rows, 'INSERT OR IGNORE INTO "link_dungeon_games" ("Dungeon_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_dungeon_games")

    cursor.execute('SELECT id, appearances FROM stg_monsters WHERE appearances IS NOT NULL')
    rows = [(monster_id, game_id) for monster_id, appearances in cursor.fetchall() for game_id in extract_guids(appearances)]
    inserted, success = execute_link_load(conn, cursor, "link_monster_games", rows, 'INSERT OR IGNORE INTO "link_monster_games" ("Monster_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_monster_games")

    cursor.execute('SELECT id, games FROM stg_items WHERE games IS NOT NULL')
    rows = [(item_id, game_id) for item_id, games in cursor.fetchall() for game_id in extract_guids(games)]
    inserted, success = execute_link_load(conn, cursor, "link_item_games", rows, 'INSERT OR IGNORE INTO "link_item_games" ("Item_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_item_games")

    cursor.execute('SELECT id, appearances FROM stg_places WHERE appearances IS NOT NULL')
    rows = [(place_id, game_id) for place_id, appearances in cursor.fetchall() for game_id in extract_guids(appearances)]
    inserted, success = execute_link_load(conn, cursor, "link_place_games", rows, 'INSERT OR IGNORE INTO "link_place_games" ("Place_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_place_games")

    cursor.execute('SELECT id, inhabitants FROM stg_places WHERE inhabitants IS NOT NULL')
    rows = [(place_id, char_id) for place_id, inhabitants in cursor.fetchall() for char_id in extract_guids(inhabitants)]
    inserted, success = execute_link_load(conn, cursor, "link_place_characters", rows, 'INSERT OR IGNORE INTO "link_place_characters" ("Place_ID", "Character_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_place_characters")

    cursor.execute('SELECT id, worked_on FROM stg_staff WHERE worked_on IS NOT NULL')
    rows = [(staff_id, game_id) for staff_id, worked_on in cursor.fetchall() for game_id in extract_guids(worked_on)]
    inserted, success = execute_link_load(conn, cursor, "link_staff_games", rows, 'INSERT OR IGNORE INTO "link_staff_games" ("Staff_ID", "Game_ID") VALUES (?, ?)')
    total_inserted += inserted
    if not success: failed_tables.append("link_staff_games")

    conn.close()

    summary = f"Pipeline complete | Total link rows inserted: {total_inserted}"
    print(summary)
    logger.info(summary)

    if failed_tables:
        warning = f"Tables that errored: {failed_tables}"
        print(warning)
        logger.warning(warning)
    else:
        success_msg = "All link tables loaded successfully"
        print(success_msg)
        logger.info(success_msg)


run_link_loads(r"C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.db")

2026-02-21 19:00:20,667 - INFO - Connected to: C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.db


2026-02-21 19:00:20,674 - INFO - link_boss_games: 419 rows already in DB | 0 rows inserted | 419 rows skipped (already existed) | 419 total rows now in DB
2026-02-21 19:00:20,677 - INFO - link_boss_dungeons: 212 rows already in DB | 0 rows inserted | 213 rows skipped (already existed) | 212 total rows now in DB
2026-02-21 19:00:20,689 - INFO - link_character_games: 1974 rows already in DB | 0 rows inserted | 1974 rows skipped (already existed) | 1974 total rows now in DB
2026-02-21 19:00:20,693 - INFO - link_dungeon_games: 441 rows already in DB | 0 rows inserted | 441 rows skipped (already existed) | 441 total rows now in DB
2026-02-21 19:00:20,705 - INFO - link_monster_games: 1710 rows already in DB | 0 rows inserted | 1710 rows skipped (already existed) | 1710 total rows now in DB
2026-02-21 19:00:20,723 - INFO - link_item_games: 2503 rows already in DB | 0 rows inserted | 2503 rows skipped (already existed) | 2503 total rows now in DB
2026-02-21 19:00:20,737 - INFO - link_place_gam

Connected to: C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.db
link_boss_games: 419 rows already in DB | 0 rows inserted | 419 rows skipped (already existed) | 419 total rows now in DB
link_boss_dungeons: 212 rows already in DB | 0 rows inserted | 213 rows skipped (already existed) | 212 total rows now in DB
link_character_games: 1974 rows already in DB | 0 rows inserted | 1974 rows skipped (already existed) | 1974 total rows now in DB
link_dungeon_games: 441 rows already in DB | 0 rows inserted | 441 rows skipped (already existed) | 441 total rows now in DB
link_monster_games: 1710 rows already in DB | 0 rows inserted | 1710 rows skipped (already existed) | 1710 total rows now in DB
link_item_games: 2503 rows already in DB | 0 rows inserted | 2503 rows skipped (already existed) | 2503 total rows now in DB
link_place_games: 1728 rows already in DB | 0 rows inserted | 1728 rows skipped (already existed) | 1728 total rows now in DB
link_place_characters: 765 rows alrea